# 03 · Activation Extraction

- **Objectives**: capture intermediate layer activations via `ActivationStore` (single forward) and `extract_representations()` (batch over a dataset), then save them to `.npz` for downstream analysis (CKA, SVCCA, probing, neural comparison).
- **Requirements**: 1 GPU (CPU fallback supported).
- **Runtime**: ~15 seconds.

## Setup

In [ ]:
import os

import numpy as np
import torch
from torch.utils.data import Dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Build a small model

In [ ]:
from kempnerforge.config.schema import ModelConfig
from kempnerforge.model.transformer import Transformer

config = ModelConfig(
    dim=128,
    n_layers=4,
    n_heads=4,
    n_kv_heads=4,
    vocab_size=256,
    max_seq_len=64,
)

model = Transformer(config).to(device)
model.eval()
print("Model ready")

## Part A · Single-forward capture with `ActivationStore`

Lowest-level API. Register hooks on a list of module FQNs, call `enable()`, run any forward pass you want, then `get(name)` to retrieve the captured tensor (on CPU).

Useful when you want complete control over the forward pass — custom prompts, specific attention masks, etc.

In [ ]:
from kempnerforge.model.hooks import ActivationStore

# FQN examples: 'layers.0.attention', 'layers.0.mlp', 'layers.3.mlp_norm', 'norm'
target_layers = ["layers.0.attention", "layers.1.mlp", "layers.3.mlp", "norm"]

store = ActivationStore(model, layers=target_layers)
store.enable()

tokens = torch.randint(0, config.vocab_size, (1, 32), device=device)
with torch.no_grad():
    _ = model(tokens)

for name in target_layers:
    act = store.get(name)
    assert act is not None, f"Expected hook to fire for {name}"
    print(f"{name:<25} shape={tuple(act.shape)}  device={act.device}  dtype={act.dtype}")

store.disable()

## Part B · Batch extraction with `extract_representations()`

Higher-level convenience wrapper. Takes a `Dataset` that yields dicts with `"input_ids"`, handles batching, puts model in eval mode, disables grad, and returns stacked activations.

Output shape: `(num_samples, seq_len, hidden_dim)` on CPU.

First we build a tiny synthetic dataset.

In [ ]:
class SyntheticTokens(Dataset):
    """Random token sequences. Yields dicts with 'input_ids', as required."""

    def __init__(self, n_samples: int, seq_len: int, vocab_size: int) -> None:
        g = torch.Generator().manual_seed(42)
        self.samples = torch.randint(0, vocab_size, (n_samples, seq_len), generator=g)

    def __len__(self) -> int:
        return self.samples.size(0)

    def __getitem__(self, idx: int) -> dict:
        return {"input_ids": self.samples[idx]}


dataset = SyntheticTokens(n_samples=128, seq_len=32, vocab_size=config.vocab_size)
print(f"Dataset: {len(dataset)} samples of shape {dataset[0]['input_ids'].shape}")

In [ ]:
from kempnerforge.model.hooks import extract_representations

acts = extract_representations(
    model,
    dataset,
    layers=["layers.1.mlp", "norm"],
    device=device,
    batch_size=16,
    max_samples=64,
)

for name, tensor in acts.items():
    print(f"{name:<20} shape={tuple(tensor.shape)}  device={tensor.device}")

## Save to `.npz` for downstream analysis

`.npz` is a standard interchange format: load in NumPy, scikit-learn, or pass to CKA/SVCCA implementations.

In [ ]:
out_path = "/tmp/kf_activations.npz"
np.savez(out_path, **{k.replace(".", "_"): v.numpy() for k, v in acts.items()})
print(f"Saved to {out_path}")

# Verify round-trip
loaded = np.load(out_path)
for k in loaded.files:
    print(f"  {k}: shape={loaded[k].shape}")

## Cleanup

In [ ]:
os.remove(out_path)
print("Removed tmp file.")

## Use cases

- **CKA / SVCCA**: compare representations between models or between layers.
- **Probing**: train a linear classifier on activations to test what information is encoded at each layer.
- **Neural data alignment (NeuroAI)**: compare model representations to recordings from biological neurons.
- **Sparse autoencoders**: train an SAE on activations to find interpretable features.